# Unknown Attack Detection and Hybrid IDS

## Objective

This notebook evaluates the proposed hybrid intrusion detection framework
for detecting unknown attacks.

The framework combines:

1. Random Forest
2. Adaptive Trust-based Ensemble Mechanism (ATEM)
3. One-Class SVM (OCSVM)

The objective is to reduce false negatives while maintaining high precision
and improving the detection of unknown attack traffic.

## Final Experimental Pipeline

Known training data
        ↓
Random Forest
        ↓
ATEM confidence-based routing
        ↓
OCSVM for uncertain samples
        ↓
Hybrid IDS decision
        ↓
Performance evaluation

In [2]:
import pandas as pd

df = pd.read_csv(
    r"E:\Hybrid_IDS_Project\dataset\CICIDS2017_Cleaned.csv"
)

print(df.shape)
print(df["Label"].value_counts())

(2520798, 79)
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [3]:
unknown_attacks = [
    "DoS slowloris",
    "DoS Slowhttptest",
    "Bot"
]

In [4]:
train_df = df[
    ~df["Label"].isin(unknown_attacks)
].copy()

In [5]:
unknown_df = df[
    df["Label"].isin(unknown_attacks)
].copy()

In [6]:
print(train_df["Label"].value_counts())

print()

print(unknown_df["Label"].value_counts())

Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
SSH-Patator                      3219
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Label
DoS slowloris       5385
DoS Slowhttptest    5228
Bot                 1948
Name: count, dtype: int64


In [7]:
train_df.to_csv(
    r"E:\Hybrid_IDS_Project\dataset\processed\train_without_unknown.csv",
    index=False
)

unknown_df.to_csv(
    r"E:\Hybrid_IDS_Project\dataset\processed\unknown_attacks.csv",
    index=False
)

In [8]:
print(train_df.shape)

print(unknown_df.shape)

print(train_df["Label"].value_counts())

print(unknown_df["Label"].value_counts())

(2508237, 79)
(12561, 79)
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
SSH-Patator                      3219
Web Attack � Brute Force         1470
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
Label
DoS slowloris       5385
DoS Slowhttptest    5228
Bot                 1948
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

In [11]:
train_known, test_known = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["Label"]
)

print(train_known.shape)
print(test_known.shape)

(2006589, 79)
(501648, 79)


In [12]:
benign_df = df[df["Label"] == "BENIGN"].copy()

In [13]:
benign_unknown = benign_df.sample(
    n=len(unknown_df),
    random_state=42
)

In [14]:
unknown_test = pd.concat(
    [unknown_df, benign_unknown],
    ignore_index=True
)

print(unknown_test.shape)
print(unknown_test["Label"].value_counts())

(25122, 79)
Label
BENIGN              12561
DoS slowloris        5385
DoS Slowhttptest     5228
Bot                  1948
Name: count, dtype: int64


In [15]:
train_known.to_csv(
    r"E:\Hybrid_IDS_Project\dataset\processed\train_known.csv",
    index=False
)

test_known.to_csv(
    r"E:\Hybrid_IDS_Project\dataset\processed\test_known.csv",
    index=False
)

unknown_test.to_csv(
    r"E:\Hybrid_IDS_Project\dataset\processed\unknown_test.csv",
    index=False
)

In [16]:
print(train_known.shape)
print(test_known.shape)
print(unknown_test.shape)

print(unknown_test["Label"].value_counts())

(2006589, 79)
(501648, 79)
(25122, 79)
Label
BENIGN              12561
DoS slowloris        5385
DoS Slowhttptest     5228
Bot                  1948
Name: count, dtype: int64


In [17]:
import pandas as pd

train_df = pd.read_csv(
    r"E:\Hybrid_IDS_Project\dataset\processed\train_known.csv"
)

test_df = pd.read_csv(
    r"E:\Hybrid_IDS_Project\dataset\processed\unknown_test.csv"
)

In [18]:
train_df["Label"] = train_df["Label"].apply(
    lambda x: 0 if x == "BENIGN" else 1
)

test_df["Label"] = test_df["Label"].apply(
    lambda x: 0 if x == "BENIGN" else 1
)

In [19]:
X_train = train_df.drop("Label", axis=1)
y_train = train_df["Label"]

X_test = test_df.drop("Label", axis=1)
y_test = test_df["Label"]

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [21]:
import numpy as np

np.save(
    r"E:\Hybrid_IDS_Project\dataset\processed\x_train_unknown.npy",
    X_train_scaled
)

np.save(
    r"E:\Hybrid_IDS_Project\dataset\processed\x_test_unknown.npy",
    X_test_scaled
)

np.save(
    r"E:\Hybrid_IDS_Project\dataset\processed\y_train_unknown.npy",
    y_train
)

np.save(
    r"E:\Hybrid_IDS_Project\dataset\processed\y_test_unknown.npy",
    y_test
)

In [22]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

print(y_train.shape)
print(y_test.shape)

(2006589, 78)
(25122, 78)
(2006589,)
(25122,)


Unknown Attack Test - Step 1: Random Forest

In [23]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

In [24]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print(rf)

RandomForestClassifier(class_weight='balanced', max_depth=20, n_jobs=-1,
                       random_state=42)


In [25]:
rf.fit(X_train_scaled, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_feat

In [26]:
joblib.dump(
    rf,
    r"E:\Hybrid_IDS_Project\dataset\models\random_forest_unknown.pkl"
)

print("Random Forest model saved successfully!")

Random Forest model saved successfully!


In [27]:
y_pred = rf.predict(X_test_scaled)

print(y_pred[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [28]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy : 0.5732027704800573
Precision: 0.9962223421478683
Recall   : 0.1469628214314147
F1 Score : 0.2561398640210906

Classification Report
              precision    recall  f1-score   support

      BENIGN       0.54      1.00      0.70     12561
      ATTACK       1.00      0.15      0.26     12561

    accuracy                           0.57     25122
   macro avg       0.77      0.57      0.48     25122
weighted avg       0.77      0.57      0.48     25122



In [29]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy : 0.5732027704800573
Precision: 0.9962223421478683
Recall   : 0.1469628214314147
F1 Score : 0.2561398640210906

Classification Report
              precision    recall  f1-score   support

      BENIGN       0.54      1.00      0.70     12561
      ATTACK       1.00      0.15      0.26     12561

    accuracy                           0.57     25122
   macro avg       0.77      0.57      0.48     25122
weighted avg       0.77      0.57      0.48     25122



In [30]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr = fp / (fp + tn)

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("False Positive Rate:", fpr)

TN: 12554
FP: 7
FN: 10715
TP: 1846
False Positive Rate: 0.0005572804713000557


In [31]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Experiment": ["Unknown Attack Test"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "FPR": [fpr],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp]
})

results.to_csv(
    r"E:\Hybrid_IDS_Project\dataset\results\unknown_attack_results.csv",
    index=False
)

print(results)

           Model           Experiment  Accuracy  Precision    Recall  \
0  Random Forest  Unknown Attack Test  0.573203   0.996222  0.146963   

   F1_Score       FPR     TN  FP     FN    TP  
0   0.25614  0.000557  12554   7  10715  1846  


In [32]:
print(y_train.value_counts() if hasattr(y_train, "value_counts") else "NumPy array")
print(np.unique(y_train, return_counts=True))

Label
0    1676045
1     330544
Name: count, dtype: int64
(array([0, 1]), array([1676045,  330544]))


In [33]:
print(np.unique(y_test, return_counts=True))

(array([0, 1]), array([12561, 12561]))


In [34]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [35]:
X_test_scaled = scaler.fit_transform(X_test)

In [36]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

print(type(X_train_scaled))
print(type(X_test_scaled))

(2006589, 78)
(25122, 78)
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [37]:
from sklearn.preprocessing import StandardScaler

print(scaler)

StandardScaler()


In [38]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [39]:
print(train_known["Label"].value_counts())

print(unknown_test["Label"].value_counts())

Label
BENIGN                        1676045
DoS Hulk                       138277
DDoS                           102411
PortScan                        72555
DoS GoldenEye                    8229
FTP-Patator                      4745
SSH-Patator                      2575
Web Attack � Brute Force         1176
Web Attack � XSS                  521
Infiltration                       29
Web Attack � Sql Injection         17
Heartbleed                          9
Name: count, dtype: int64
Label
BENIGN              12561
DoS slowloris        5385
DoS Slowhttptest     5228
Bot                  1948
Name: count, dtype: int64


In [40]:
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib
import pandas as pd

In [41]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42
)

print(mlp)

MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=50, random_state=42)


In [42]:
mlp.fit(X_train_scaled, y_train)

e:\Hybrid_IDS_Project\dataset\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",50
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True


In [43]:
joblib.dump(
    mlp,
    r"E:\Hybrid_IDS_Project\dataset\models\mlp_unknown.pkl"
)

print("MLP model saved successfully!")

MLP model saved successfully!


In [44]:
y_pred = mlp.predict(X_test_scaled)

In [45]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy : 0.5562853276013057
Precision: 0.9971870604781997
Recall   : 0.11288910118621129
F1 Score : 0.20281770721590503

Classification Report
              precision    recall  f1-score   support

      BENIGN       0.53      1.00      0.69     12561
      ATTACK       1.00      0.11      0.20     12561

    accuracy                           0.56     25122
   macro avg       0.76      0.56      0.45     25122
weighted avg       0.76      0.56      0.45     25122



In [46]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr = fp / (fp + tn)

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("False Positive Rate:", fpr)

TN: 12557
FP: 4
FN: 11143
TP: 1418
False Positive Rate: 0.0003184459836000318


In [47]:
new_result = pd.DataFrame({
    "Model": ["MLP"],
    "Experiment": ["Unknown Attack Test"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "FPR": [fpr],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp]
})

file_path = r"E:\Hybrid_IDS_Project\dataset\results\unknown_attack_results.csv"

results = pd.read_csv(file_path)
results = pd.concat([results, new_result], ignore_index=True)

results.to_csv(file_path, index=False)

print(results)

           Model           Experiment  Accuracy  Precision    Recall  \
0  Random Forest  Unknown Attack Test  0.573203   0.996222  0.146963   
1            MLP  Unknown Attack Test  0.556285   0.997187  0.112889   

   F1_Score       FPR     TN  FP     FN    TP  
0  0.256140  0.000557  12554   7  10715  1846  
1  0.202818  0.000318  12557   4  11143  1418  


In [48]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import pandas as pd

In [49]:
X_train_cnn = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_cnn = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

print(X_train_cnn.shape)
print(X_test_cnn.shape)

(2006589, 78, 1)
(25122, 78, 1)


In [50]:
import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

In [51]:
cnn = Sequential([

    Input(shape=(78,1)),

    Conv1D(
        filters=32,
        kernel_size=3,
        activation='relu'
    ),

    MaxPooling1D(pool_size=2),

    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu'
    ),

    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(64, activation='relu'),

    Dense(1, activation='sigmoid')
])

In [52]:
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [53]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 76, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 38, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 36, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 162,241 (633.75 KB)

 Trainable params: 162,241 (633.75 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print(class_weight_dict)

{np.int64(0): np.float64(0.5986083309219025), np.int64(1): np.float64(3.035282746018684)}


In [55]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [56]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [57]:
checkpoint = ModelCheckpoint(
    r"E:\Hybrid_IDS_Project\dataset\models\cnn_unknown.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [58]:
history = cnn.fit(
    X_train_cnn,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=1024,
    class_weight=class_weight_dict,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

Epoch 1/20
1566/1568 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9621 - loss: 0.0759
Epoch 1: val_loss improved from None to 0.04836, saving model to E:\Hybrid_IDS_Project\dataset\models\cnn_unknown.keras

Epoch 1: finished saving model to E:\Hybrid_IDS_Project\dataset\models\cnn_unknown.keras
1568/1568 ━━━━━━━━━━━━━━━━━━━━ 29s 18ms/step - accuracy: 0.9621 - loss: 0.0759 - val_accuracy: 0.9751 - val_loss: 0.0484
Epoch 2/20
1565/1568 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9719 - loss: 0.0485
Epoch 2: val_loss improved from 0.04836 to 0.04516, saving model to E:\Hybrid_IDS_Project\dataset\models\cnn_unknown.keras

Epoch 2: finished saving model to E:\Hybrid_IDS_Project\dataset\models\cnn_unknown.keras
1568/1568 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - accuracy: 0.9719 - loss: 0.0485 - val_accuracy: 0.9777 - val_loss: 0.0452
Epoch 3/20
1566/1568 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9739 - loss: 0.0444
Epoch 3: val_loss did not improve from 0.04516
1568/1568 ━━━━━━━━━━━

In [59]:
from tensorflow.keras.models import load_model

cnn = load_model(
    r"E:\Hybrid_IDS_Project\dataset\models\cnn_unknown.keras"
)

In [60]:
y_prob = cnn.predict(X_test_cnn)

y_pred = (y_prob > 0.5).astype(int).flatten()

786/786 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [61]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy : 0.5817212005413581
Precision: 0.9867235656709341
Recall   : 0.16567152296791657
F1 Score : 0.28370824812542605

Classification Report
              precision    recall  f1-score   support

      BENIGN       0.54      1.00      0.70     12561
      ATTACK       0.99      0.17      0.28     12561

    accuracy                           0.58     25122
   macro avg       0.77      0.58      0.49     25122
weighted avg       0.77      0.58      0.49     25122



In [62]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr = fp / (fp + tn)

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("False Positive Rate:", fpr)

TN: 12533
FP: 28
FN: 10480
TP: 2081
False Positive Rate: 0.0022291218852002227


In [63]:
import pandas as pd

new_result = pd.DataFrame({
    "Model": ["CNN"],
    "Experiment": ["Unknown Attack Test"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "FPR": [fpr],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp]
})

file_path = r"E:\Hybrid_IDS_Project\dataset\results\unknown_attack_results.csv"

results = pd.read_csv(file_path)
results = pd.concat([results, new_result], ignore_index=True)

results.to_csv(file_path, index=False)

print(results)

           Model           Experiment  Accuracy  Precision    Recall  \
0  Random Forest  Unknown Attack Test  0.573203   0.996222  0.146963   
1            MLP  Unknown Attack Test  0.556285   0.997187  0.112889   
2            CNN  Unknown Attack Test  0.581721   0.986724  0.165672   

   F1_Score       FPR     TN  FP     FN    TP  
0  0.256140  0.000557  12554   7  10715  1846  
1  0.202818  0.000318  12557   4  11143  1418  
2  0.283708  0.002229  12533  28  10480  2081  


OCSVM (Unknown Attack Test)


In [64]:
X_train_normal = X_train_scaled[y_train == 0]

print(X_train_normal.shape)

(1676045, 78)


In [65]:
from sklearn.utils import resample

X_train_sample = resample(
    X_train_normal,
    n_samples=50000,
    random_state=42
)

print(X_train_sample.shape)

(50000, 78)


In [66]:
from sklearn.svm import OneClassSVM

In [67]:
OneClassSVM(
    kernel='rbf',
    gamma='scale',
    nu=0.05
)

,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.05
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [68]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.05
)

print(ocsvm)

OneClassSVM(nu=0.05)


In [69]:
ocsvm.fit(X_train_sample)

,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.05
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1
Name,Type,Value


In [70]:
import joblib

joblib.dump(
    ocsvm,
    r"E:\Hybrid_IDS_Project\dataset\models\ocsvm_unknown.pkl"
)

print("OCSVM model saved successfully!")

OCSVM model saved successfully!


In [71]:
y_pred = ocsvm.predict(X_test_scaled)

In [72]:
import numpy as np

y_pred = np.where(y_pred == 1, 0, 1)

In [73]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy : 0.7963935992357296
Precision: 0.9278326821420363
Recall   : 0.6427832178966643
F1 Score : 0.7594412829798242

Classification Report
              precision    recall  f1-score   support

      BENIGN       0.73      0.95      0.82     12561
      ATTACK       0.93      0.64      0.76     12561

    accuracy                           0.80     25122
   macro avg       0.83      0.80      0.79     25122
weighted avg       0.83      0.80      0.79     25122



In [74]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr = fp / (fp + tn)

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("False Positive Rate:", fpr)

TN: 11933
FP: 628
FN: 4487
TP: 8074
False Positive Rate: 0.049996019425205


In [75]:
import pandas as pd

new_result = pd.DataFrame({
    "Model": ["OCSVM"],
    "Experiment": ["Unknown Attack Test"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "FPR": [fpr],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp]
})

file_path = r"E:\Hybrid_IDS_Project\dataset\results\unknown_attack_results.csv"

results = pd.read_csv(file_path)
results = pd.concat([results, new_result], ignore_index=True)

results.to_csv(file_path, index=False)

print(results)

           Model           Experiment  Accuracy  Precision    Recall  \
0  Random Forest  Unknown Attack Test  0.573203   0.996222  0.146963   
1            MLP  Unknown Attack Test  0.556285   0.997187  0.112889   
2            CNN  Unknown Attack Test  0.581721   0.986724  0.165672   
3          OCSVM  Unknown Attack Test  0.796394   0.927833  0.642783   

   F1_Score       FPR     TN   FP     FN    TP  
0  0.256140  0.000557  12554    7  10715  1846  
1  0.202818  0.000318  12557    4  11143  1418  
2  0.283708  0.002229  12533   28  10480  2081  
3  0.759441  0.049996  11933  628   4487  8074  


LOF (Unknown Attack Test)

In [76]:
from sklearn.neighbors import LocalOutlierFactor

In [77]:
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    novelty=True
)

In [78]:
lof.fit(X_train_sample)

,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. When fitting this is used to define thethreshold on the scores of the samples.- if 'auto', the threshold is determined as in the original paper,- if a float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"novelty novelty: bool, default=FalseBy default, LocalOutlierFactor is only meant to be used for outlierdetection (novelty=False). Set novelty to True if you want to useLocalOutlierFactor for novelty detection. In this case be aware thatyou should only use predict, decision_function and score_sampleson new unseen data and not on the training set; and note that theresults obtained this way may differ from the standard LOF results... versionadded:: 0.20",True
,"n_neighbors n_neighbors: int, default=20Number of neighbors to use by default for :meth:`kneighbors` queries.If n_neighbors is larger than the number of samples provided,all samples will be used.",20
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf is size passed to :class:`BallTree` or :class:`KDTree`. This canaffect the speed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float, default=2Parameter for the Minkowski metric from:func:`sklearn.metrics.pairwise_distances`. When p = 1, thisis equivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strThe effective metric used for the distance computation.,str,'eu...an'


In [79]:
y_pred = lof.predict(X_test_scaled)

import numpy as np

# Convert predictions
#  1  -> BENIGN (0)
# -1 -> ATTACK (1)
y_pred = np.where(y_pred == 1, 0, 1)

In [80]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

Accuracy : 0.5521853355624552
Precision: 0.752991123118487
Recall   : 0.15532202850091553
F1 Score : 0.25752375923970433

Classification Report
              precision    recall  f1-score   support

      BENIGN       0.53      0.95      0.68     12561
      ATTACK       0.75      0.16      0.26     12561

    accuracy                           0.55     25122
   macro avg       0.64      0.55      0.47     25122
weighted avg       0.64      0.55      0.47     25122



In [82]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

fpr = fp / (fp + tn)

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("False Positive Rate:", fpr)

TN: 11921
FP: 640
FN: 10610
TP: 1951
False Positive Rate: 0.0509513573760051


In [86]:
import pandas as pd

new_result = pd.DataFrame({
    "Model": ["LOF"],
    "Experiment": ["Unknown Attack Test"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1_Score": [f1],
    "FPR": [fpr],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp]
})

file_path = r"E:\Hybrid_IDS_Project\dataset\results\unknown_attack_results.csv"

results = pd.read_csv(file_path)
results = pd.concat([results, new_result], ignore_index=True)

results.to_csv(file_path, index=False)

print(results)

           Model           Experiment  Accuracy  Precision    Recall  \
0  Random Forest  Unknown Attack Test  0.573203   0.996222  0.146963   
1            MLP  Unknown Attack Test  0.556285   0.997187  0.112889   
2            CNN  Unknown Attack Test  0.581721   0.986724  0.165672   
3          OCSVM  Unknown Attack Test  0.796394   0.927833  0.642783   
4            LOF  Unknown Attack Test  0.552185   0.752991  0.155322   
5            LOF  Unknown Attack Test  0.552185   0.752991  0.155322   

   F1_Score       FPR     TN   FP     FN    TP  
0  0.256140  0.000557  12554    7  10715  1846  
1  0.202818  0.000318  12557    4  11143  1418  
2  0.283708  0.002229  12533   28  10480  2081  
3  0.759441  0.049996  11933  628   4487  8074  
4  0.257524  0.050951  11921  640  10610  1951  
5  0.257524  0.050951  11921  640  10610  1951  


In [87]:
joblib.dump(rf_model, "models/random_forest_unknown.pkl")

NameError: name 'rf_model' is not defined